# Save model outputs, then judge them with Gemini

1. Generate summaries from the checkpoints on Drive and save them to Drive (`generate_outputs.py`, needs GPU).
2. Score and compare them with Gemini (`llm_judge.py`, no GPU needed).

The checkpoints are read directly from Drive, nothing has to be copied into `./cache`.

In [ ]:
# Install condacolab. The runtime restarts after this cell, that is normal.
!pip install -q condacolab
import condacolab
condacolab.install()

### Env

In [ ]:
# Clone the repo and mount Google Drive.
!git clone https://github.com/alvrian/BRIO.git

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

%cd BRIO

In [ ]:
# Same env setup as BRIO_1_1.ipynb, then unzip the fine-tuned IndoBART baseline.
# The preprocessed dataset zip is not needed here, only test.source / test.target from Drive.
! conda create --name env --file spec-file.txt
! conda run --no-capture-output -n env pip install -r requirements.txt
! conda run -n env python -c "import nltk;nltk.download('punkt')"
! git clone https://github.com/neulab/compare-mt.git
! cd compare-mt && conda run --no-capture-output -n env pip install -r requirements.txt
! cd compare-mt && conda run -n env python setup.py install
print("done setting up env")
! cp /content/drive/MyDrive/BRIO/indobart-liputan6-finetuned.zip .
! unzip -q indobart-liputan6-finetuned.zip
! rm indobart-liputan6-finetuned.zip
print("all done")

### Paths

In [ ]:
# List every model_generation.bin saved on Drive, to copy the right path into the next cell.
import glob
for p in sorted(glob.glob("/content/drive/MyDrive/BRIO/checkpoints/*/*/model_generation.bin")):
    print(p)

In [ ]:
import os

# Liputan6 canonical test set (same files the eval cells in BRIO_1_1.ipynb copy from Drive).
TEST_SOURCE = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.source"
TEST_TARGET = "/content/drive/MyDrive/BRIO/liputan6_candidates/canonical/test.target"

# Fine-tuned IndoBART baseline, unzipped by the env cell.
BASELINE = "./indobart-liputan6-finetuned"

# BRIO checkpoints to evaluate: short name -> path of model_generation.bin.
# EDIT: keep the ones you want. Each extra checkpoint costs 100 more Gemini requests.
CKPT_DIR = "/content/drive/MyDrive/BRIO/checkpoints"
BRIO_CKPTS = {
    "brio": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_second_seed/master_1_1/model_generation.bin",
    # "brio_seed3": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_third_seed/master_1_1/model_generation.bin",
    # "brio_cand12": CKPT_DIR + "/BRIO_Liputan6_checkpoint_1-1_12/master_1_1/model_generation.bin",
}

# Folder on Drive where the summaries and the judge results are saved.
OUT_DIR = "/content/drive/MyDrive/BRIO/eval_outputs"

# Number of test articles (the same articles for every model). 0 = whole test set.
SAMPLE_N = 100

for p in [TEST_SOURCE, TEST_TARGET, BASELINE] + list(BRIO_CKPTS.values()):
    print("found  " if os.path.exists(p) else "MISSING", p)

### 1. Generate and save the summaries

In [ ]:
# Baseline summaries -> OUT_DIR/baseline.out
# The first run also picks the articles and saves sample_ids.txt, sample.source, sample.target.
!conda run --no-capture-output -n env python generate_outputs.py --name baseline \
    --model_path "{BASELINE}" --src_file "{TEST_SOURCE}" --tgt_file "{TEST_TARGET}" \
    --out_dir "{OUT_DIR}" --sample_n {SAMPLE_N}

In [ ]:
# BRIO summaries on the same articles -> OUT_DIR/<name>.out, one file per checkpoint.
for name, ckpt in BRIO_CKPTS.items():
    print("=====", name)
    !conda run --no-capture-output -n env python generate_outputs.py --name {name} --model_path "{ckpt}" --src_file "{TEST_SOURCE}" --tgt_file "{TEST_TARGET}" --out_dir "{OUT_DIR}" --sample_n {SAMPLE_N}

In [ ]:
# Quick look at the first picked article: article, reference, then each model's summary.
for name in ["sample.source", "sample.target", "baseline.out"] + [n + ".out" for n in BRIO_CKPTS]:
    with open(os.path.join(OUT_DIR, name), encoding="utf-8") as f:
        print(name, "->", f.readline().strip()[:400], "\n")

### 2. Judge with Gemini

Add your key in Colab: key icon on the left (Secrets) -> name `GEMINI_API_KEY` -> allow notebook access.

Requests = articles x systems + articles x pairs. With 100 articles, baseline + one BRIO + reference and one pair: 400 requests.

In [ ]:
# Read the API key from Colab Secrets and build the judge command once.
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Score baseline, every BRIO checkpoint and the gold reference; compare each BRIO against the baseline.
systems = " ".join(["baseline"] + list(BRIO_CKPTS))
pairs = " ".join("--pair baseline " + n for n in BRIO_CKPTS)
JUDGE = 'conda run --no-capture-output -n env python llm_judge.py --data_dir "%s" --systems %s %s --include_reference' % (OUT_DIR, systems, pairs)
print(JUDGE)

In [ ]:
# Dry run: shows how many requests are needed and one example prompt. No API call.
!{JUDGE} --dry_run

In [ ]:
# Small test on 3 articles first to check that the key and the model work.
!{JUDGE} --limit 3

In [ ]:
# Full run. Stops by itself at 450 requests or when the quota is hit.
# Results are saved after every answer: run this cell again the next day to continue.
# --human_n 30 also writes a blind rating sheet for human raters (human_eval_sheet.csv).
!{JUDGE} --human_n 30

In [ ]:
# Show the result tables.
import pandas as pd
print(open(os.path.join(OUT_DIR, "judge", "summary.txt"), encoding="utf-8").read())
scores = pd.read_csv(os.path.join(OUT_DIR, "judge", "scores.csv"))
scores.head()